In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Geometry-V4 G0 fixed SD3.5 Colab handoff
This notebook clones the published Geometry-V4 branch, checks out one detached exact, and runs only the frozen four-unit G0 roster. Drive is a create-only result sink, never a source input. G1 is not in the default execution path.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, subprocess, sys

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Geometry-V4'
SOURCE_EXACT = '6f819f9051f11305b6d07a810c4d6fe328afdc2e'
REPO = Path('/content/cegwm-geometry-v4-g0-g1-source')
DRIVE_RUNS = Path('/content/drive/MyDrive/CEG-WM/Geometry-V4-G0-G1/runs')

if REPO.exists():
    raise FileExistsError('fresh Colab runtime required; source checkout already exists')
subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, capture_output=True, text=True).stdout.strip()

CLONED_BRANCH = git('branch', '--show-current')
subprocess.run(['git', 'checkout', '--detach', SOURCE_EXACT], cwd=REPO, check=True)
assert git('rev-parse', 'HEAD') == SOURCE_EXACT
assert git('branch', '--show-current') == ''
assert git('status', '--porcelain') == ''
CONFIG = json.loads((REPO/'configs/geometry_v4/geometry_v4_g0_g1_v1.json').read_text(encoding='ascii'))
CONFIG_SHA256 = hashlib.sha256((REPO/'configs/geometry_v4/geometry_v4_g0_g1_v1.json').read_bytes()).hexdigest()
RUN_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ROOT = DRIVE_RUNS / f'{SOURCE_EXACT}-{RUN_UTC}'
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
if RUN_ROOT.exists():
    raise FileExistsError('create-only run directory already exists')
print({'repo_url': REPO_URL, 'cloned_branch': CLONED_BRANCH, 'source_exact': SOURCE_EXACT, 'detached': True, 'clean': True, 'config_sha256': CONFIG_SHA256, 'model': CONFIG['identity']['model_id'], 'placement': CONFIG['identity']['placement'], 'steps': 20, 'dtype': 'float16', 'budget': CONFIG['residual_budget'], 'g0_roster': CONFIG['g0'], 'g1_roster': CONFIG['g1'], 'drive_run_root': str(RUN_ROOT)})


In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU runtime is required; no G0 record may be created on CPU'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_BYTES = torch.cuda.get_device_properties(0).total_memory
print({'cuda_available': True, 'gpu_name': GPU_NAME, 'vram_bytes': GPU_VRAM_BYTES})
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
from google.colab import userdata
from cegwm.runtime.content_weighted_joint_sd35 import derive_stability_wrong_keys
from cegwm.shared.keys import normalize_detection_key
from experiments.geometry_v4_generative_engine import run
HF_TOKEN = userdata.get('HF_TOKEN')
DETECTION_KEY = userdata.get('CEG_WM_ROOT_KEY')
assert all(isinstance(value, str) and value.strip() for value in (HF_TOKEN, DETECTION_KEY))
NORMALIZED_KEY = normalize_detection_key(DETECTION_KEY)
WRONG_KEY = derive_stability_wrong_keys(NORMALIZED_KEY)[0]
assert isinstance(WRONG_KEY, bytes) and WRONG_KEY != NORMALIZED_KEY
# Exactly one fixed G0 invocation; the runner reuses the frozen content weighted-joint detector.
g0_records = run('G0', DETECTION_KEY, WRONG_KEY, repo_root=REPO, hf_token=HF_TOKEN, artifact_root=RUN_ROOT)
run_metadata = {'repo_url': REPO_URL, 'branch': CLONED_BRANCH, 'source_exact': SOURCE_EXACT, 'detached': True, 'clean': True, 'config_sha256': CONFIG_SHA256, 'model': CONFIG['identity']['model_id'], 'placement': CONFIG['identity']['placement'], 'steps': 20, 'dtype': 'float16', 'budget': CONFIG['residual_budget'], 'g0_roster': CONFIG['g0'], 'g1_roster': CONFIG['g1'], 'gpu': {'name': GPU_NAME, 'vram_bytes': GPU_VRAM_BYTES}, 'content_detector': CONFIG['content_detector']}
metadata_path = RUN_ROOT / 'colab_run_metadata.json'
if metadata_path.exists():
    raise FileExistsError('create-only metadata path already exists')
metadata_path.write_text(json.dumps(run_metadata, sort_keys=True, indent=2) + '\n', encoding='ascii')
print({'g0_units': len(g0_records), 'g0_passed': sum(record['final_rgb'] is not None and record['final_rgb']['passed'] for record in g0_records), 'metadata_sha256': hashlib.sha256(metadata_path.read_bytes()).hexdigest()})
# Do not run G1 here. After an independent G0 4/4 freeze, use a new create-only UTC RUN_ROOT and execute G1 exactly once.
